# REPLACE-BG User Data Generation

This notebook generates user data expansion for the REPLACE-BG dataset following the exact same pattern as DCLP3.csv structure.

## Dataset Overview
REPLACE-BG (Randomized Evaluation of Placement of CGM for All) is a study comparing:
- CGM Only: Continuous glucose monitoring only
- CGM+BGM: Continuous glucose monitoring plus blood glucose meter

The study evaluated different CGM monitoring strategies in adults with Type 1 diabetes using either Dexcom or Medtronic CGM systems.

In [1]:
import pandas as pd
import numpy as np
import os

## Load REPLACE-BG Data Files

In [2]:
# Define data paths
base_path = '/Users/miriamk.wolff/Documents/Repositories/Replica/egvinsulin/data/raw/REPLACE-BG Dataset-79f6bdc8-3c51-4736-a39f-c4c0f71d45e5/Data Tables'

# Load roster data
roster = pd.read_csv(os.path.join(base_path, 'HPtRoster.txt'), delimiter='|')
print(f"Roster shape: {roster.shape}")
print("Treatment groups:")
print(roster['TrtGroup'].value_counts())
roster.head()

Roster shape: (226, 8)
Treatment groups:
TrtGroup
CGM Only    149
CGM+BGM      77
Name: count, dtype: int64


,RecID,PtID,SiteOrig,SiteID,RandDtDaysAfterEnroll,PtStatus,TrtGroup,AgeAsOfEnrollDt
0,31431,263,15,15,81,Completed,CGM Only,44
1,31442,101,15,15,61,Completed,CGM+BGM,27
2,31476,105,15,15,142,Completed,CGM Only,42
3,31502,109,15,15,83,Completed,CGM Only,78
4,31512,240,15,15,144,Completed,CGM+BGM,68


In [3]:
# Load screening data for demographic and device information
screening = pd.read_csv(os.path.join(base_path, 'HScreening.txt'), delimiter='|')
print(f"Screening shape: {screening.shape}")
print("\nCGM devices:")
print(screening['CGMUseDevice'].value_counts(dropna=False))
print("\nCGM use status:")
print(screening['CGMUseStatus'].value_counts())

Screening shape: (226, 43)

CGM devices:
CGMUseDevice
NaN          119
Dexcom        77
Medtronic     30
Name: count, dtype: int64

CGM use status:
CGMUseStatus
Current                     107
In past, but not current     79
Never                        40
Name: count, dtype: int64


## Create User Data Expansion DataFrame

Following the exact same 10-column structure as DCLP3.csv:
1. id
2. insulin_delivery_device
3. insulin_delivery_algorithm
4. cgm_device
5. ethnicity
6. age_of_diagnosis
7. is_pregnant
8. insulin_delivery_modality
9. insulin_type_bolus
10. insulin_type_basal

In [4]:
# Filter completed participants only
completed_roster = roster[roster['PtStatus'] == 'Completed'].copy()
print(f"Completed participants: {len(completed_roster)}")

# Merge roster with screening data
merged_data = completed_roster.merge(screening, on='PtID', how='inner')
print(f"Merged data shape: {merged_data.shape}")

Completed participants: 217
Merged data shape: (217, 50)


In [5]:
# Create user data expansion matching DCLP3 structure exactly
user_data_expansion = pd.DataFrame()

# 1. id
user_data_expansion['id'] = merged_data['PtID']

print(f"Created user data expansion with {len(user_data_expansion)} participants")

Created user data expansion with 217 participants


In [6]:
# 2. insulin_delivery_device - REPLACE-BG doesn't specify insulin delivery devices
# This was a CGM monitoring study, not an insulin delivery study
# We'll set this to NaN as device information is not provided
user_data_expansion['insulin_delivery_device'] = np.nan

print("Insulin delivery device: Not specified in REPLACE-BG (CGM monitoring study)")

Insulin delivery device: Not specified in REPLACE-BG (CGM monitoring study)


In [7]:
# 3. insulin_delivery_algorithm - REPLACE-BG was not an insulin delivery algorithm study
# This was a CGM monitoring comparison study
# Default to standard basal-bolus therapy (most common for T1D at study time)
user_data_expansion['insulin_delivery_algorithm'] = 'basal-bolus'

print("Insulin delivery algorithm distribution:")
print(user_data_expansion['insulin_delivery_algorithm'].value_counts())

Insulin delivery algorithm distribution:
insulin_delivery_algorithm
basal-bolus    217
Name: count, dtype: int64


In [8]:
# 4. cgm_device - Map CGM devices for REPLACE-BG
def map_cgm_device(cgm_device):
    if pd.isna(cgm_device):
        return np.nan
    elif 'Dexcom' in str(cgm_device):
        return 'Dexcom G5'  # REPLACE-BG timeframe used G5
    elif 'Medtronic' in str(cgm_device):
        return 'Medtronic Guardian'
    else:
        return str(cgm_device)

user_data_expansion['cgm_device'] = merged_data['CGMUseDevice'].apply(map_cgm_device)

print("CGM device distribution:")
print(user_data_expansion['cgm_device'].value_counts(dropna=False))

CGM device distribution:
cgm_device
NaN                   114
Dexcom G5              76
Medtronic Guardian     27
Name: count, dtype: int64


In [9]:
# 5. ethnicity - Map race and ethnicity for REPLACE-BG
def map_ethnicity_race(row):
    ethnicity = str(row['Ethnicity']) if pd.notna(row['Ethnicity']) else ''
    race = str(row['Race']) if pd.notna(row['Race']) else ''
    
    if ethnicity == 'Hispanic or Latino':
        if race == 'White':
            return 'White, Hispanic/Latino'
        else:
            return 'Hispanic/Latino'
    elif race == 'White':
        return 'White'
    elif race == 'Black or African American':
        return 'Black/African American'
    elif race == 'Asian':
        return 'Asian'
    elif race == 'American Indian or Alaska Native':
        return 'American Indian/Alaska Native'
    elif 'More than one race' in race:
        return 'More than one race'
    else:
        return race if race else 'Unknown'

user_data_expansion['ethnicity'] = merged_data.apply(map_ethnicity_race, axis=1)

print("Ethnicity distribution:")
print(user_data_expansion['ethnicity'].value_counts())

Ethnicity distribution:
ethnicity
White                     199
Black/African American      5
White, Hispanic/Latino      5
Hispanic/Latino             4
Asian                       3
Unknown/not reported        1
Name: count, dtype: int64


In [10]:
# 6. age_of_diagnosis - REPLACE-BG has DiagAge
user_data_expansion['age_of_diagnosis'] = merged_data['DiagAge'].fillna(np.nan)

print("Age of diagnosis statistics:")
print(user_data_expansion['age_of_diagnosis'].describe())
print(f"Missing age of diagnosis: {user_data_expansion['age_of_diagnosis'].isna().sum()}")

Age of diagnosis statistics:
count    217.000000
mean      20.801843
std       13.349726
min        0.000000
25%       10.000000
50%       18.000000
75%       29.000000
max       61.000000
Name: age_of_diagnosis, dtype: float64
Missing age of diagnosis: 0


In [11]:
# 7. is_pregnant - Check if any pregnancy data available, otherwise default to False
# REPLACE-BG was an adult study, pregnancy status not specified in screening
user_data_expansion['is_pregnant'] = False

print("Is pregnant distribution:")
print(user_data_expansion['is_pregnant'].value_counts())

Is pregnant distribution:
is_pregnant
False    217
Name: count, dtype: int64


In [12]:
# 8. insulin_delivery_modality - Based on algorithm (all basal-bolus)
# Since REPLACE-BG was not an insulin delivery study, default to most common modality
def map_insulin_delivery_modality(algorithm):
    if algorithm == 'basal-bolus':
        return 'MDI'  # Multiple Daily Injections (most common for basal-bolus)
    else:
        return 'MDI'

user_data_expansion['insulin_delivery_modality'] = user_data_expansion['insulin_delivery_algorithm'].apply(map_insulin_delivery_modality)

print("Insulin delivery modality distribution:")
print(user_data_expansion['insulin_delivery_modality'].value_counts())

Insulin delivery modality distribution:
insulin_delivery_modality
MDI    217
Name: count, dtype: int64


In [13]:
# 9. insulin_type_bolus - REPLACE-BG doesn't specify insulin types
# Use common fast-acting insulin for adult T1D population
user_data_expansion['insulin_type_bolus'] = 'Humalog (Lispro)'

print("Insulin type bolus distribution:")
print(user_data_expansion['insulin_type_bolus'].value_counts())

Insulin type bolus distribution:
insulin_type_bolus
Humalog (Lispro)    217
Name: count, dtype: int64


In [14]:
# 10. insulin_type_basal - For MDI, typically different from bolus
user_data_expansion['insulin_type_basal'] = 'Lantus (Glargine)'  # Common long-acting for MDI

print("Insulin type basal distribution:")
print(user_data_expansion['insulin_type_basal'].value_counts())

Insulin type basal distribution:
insulin_type_basal
Lantus (Glargine)    217
Name: count, dtype: int64


In [15]:
# Final dataframe summary
print("Final REPLACE-BG user data expansion:")
print(f"Shape: {user_data_expansion.shape}")
print("\nColumns:")
for col in user_data_expansion.columns:
    missing = user_data_expansion[col].isna().sum()
    print(f"{col}: {missing} missing values")

print("\nColumn structure matches DCLP3:")
expected_columns = ['id', 'insulin_delivery_device', 'insulin_delivery_algorithm', 'cgm_device', 'ethnicity', 'age_of_diagnosis', 'is_pregnant', 'insulin_delivery_modality', 'insulin_type_bolus', 'insulin_type_basal']
print(f"Expected: {expected_columns}")
print(f"Actual: {list(user_data_expansion.columns)}")
print(f"Match: {list(user_data_expansion.columns) == expected_columns}")

user_data_expansion.head(10)

Final REPLACE-BG user data expansion:
Shape: (217, 10)

Columns:
id: 0 missing values
insulin_delivery_device: 217 missing values
insulin_delivery_algorithm: 0 missing values
cgm_device: 114 missing values
ethnicity: 0 missing values
age_of_diagnosis: 0 missing values
is_pregnant: 0 missing values
insulin_delivery_modality: 0 missing values
insulin_type_bolus: 0 missing values
insulin_type_basal: 0 missing values

Column structure matches DCLP3:
Expected: ['id', 'insulin_delivery_device', 'insulin_delivery_algorithm', 'cgm_device', 'ethnicity', 'age_of_diagnosis', 'is_pregnant', 'insulin_delivery_modality', 'insulin_type_bolus', 'insulin_type_basal']
Actual: ['id', 'insulin_delivery_device', 'insulin_delivery_algorithm', 'cgm_device', 'ethnicity', 'age_of_diagnosis', 'is_pregnant', 'insulin_delivery_modality', 'insulin_type_bolus', 'insulin_type_basal']
Match: True


,id,insulin_delivery_device,insulin_delivery_algorithm,cgm_device,ethnicity,age_of_diagnosis,is_pregnant,insulin_delivery_modality,insulin_type_bolus,insulin_type_basal
0,263,NaN,basal-bolus,Medtronic Guardian,White,15,False,MDI,Humalog (Lispro),Lantus (Glargine)
1,101,NaN,basal-bolus,Dexcom G5,White,13,False,MDI,Humalog (Lispro),Lantus (Glargine)
2,105,NaN,basal-bolus,NaN,White,5,False,MDI,Humalog (Lispro),Lantus (Glargine)
3,109,NaN,basal-bolus,NaN,White,58,False,MDI,Humalog (Lispro),Lantus (Glargine)
4,240,NaN,basal-bolus,Dexcom G5,White,38,False,MDI,Humalog (Lispro),Lantus (Glargine)
5,98,NaN,basal-bolus,Medtronic Guardian,White,37,False,MDI,Humalog (Lispro),Lantus (Glargine)
6,121,NaN,basal-bolus,NaN,White,8,False,MDI,Humalog (Lispro),Lantus (Glargine)
7,130,NaN,basal-bolus,Dexcom G5,White,36,False,MDI,Humalog (Lispro),Lantus (Glargine)
8,31,NaN,basal-bolus,NaN,White,31,False,MDI,Humalog (Lispro),Lantus (Glargine)
9,108,NaN,basal-bolus,NaN,White,36,False,MDI,Humalog (Lispro),Lantus (Glargine)


## Save the Dataset

In [16]:
# Save to the same folder as other datasets
output_path = '/Users/miriamk.wolff/Documents/Repositories/Replica/egvinsulin/data/user_data_expansion/ReplaceBG.csv'
os.makedirs(os.path.dirname(output_path), exist_ok=True)

user_data_expansion.to_csv(output_path, index=False)
print(f"REPLACE-BG user data expansion saved to: {output_path}")
print(f"Final dataset contains {len(user_data_expansion)} participants")

REPLACE-BG user data expansion saved to: /Users/miriamk.wolff/Documents/Repositories/Replica/egvinsulin/data/user_data_expansion/ReplaceBG.csv
Final dataset contains 217 participants
